In [1]:
from pathlib import Path
import pandas as pd

# Resolve data directory whether running from repo root or food_data folder
cwd = Path.cwd().resolve()
data_dir = cwd / "food_data"
if not data_dir.exists():
    data_dir = cwd

# Load the CSV files separately
food = pd.read_csv(data_dir / "food.csv")
food_nutrient = pd.read_csv(data_dir / "food_nutrient.csv", low_memory=False) # low_memory=False avoids mixed-type chunk inference warnings for large CSVs
nutrient = pd.read_csv(data_dir / "nutrient.csv")
food_category = pd.read_csv(data_dir / "food_category.csv")
foundation_food = pd.read_csv(data_dir / "foundation_food.csv")

# Quick check
summary = pd.DataFrame(
    {
        "table": ["food", "food_nutrient", "nutrient", "food_category", "foundation_food"],
        "rows": [len(food), len(food_nutrient), len(nutrient), len(food_category), len(foundation_food)],
        "columns": [food.shape[1], food_nutrient.shape[1], nutrient.shape[1], food_category.shape[1], foundation_food.shape[1]],
    }
)
display(summary)

# Example preview
display(food.head())
display(food_nutrient.head())
display(nutrient.head())
display(food_category.head())
display(foundation_food.head())

,table,rows,columns
0,food,78026,5
1,food_nutrient,159285,11
2,nutrient,477,5
3,food_category,28,3
4,foundation_food,365,3


,fdc_id,data_type,description,food_category_id,publication_date
0,319874,sample_food,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
1,319875,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
2,319876,market_acquisition,"HUMMUS, SABRA CLASSIC",16.0,2019-04-01
3,319877,sub_sample_food,Hummus,16.0,2019-04-01
4,319878,sub_sample_food,Hummus,16.0,2019-04-01


,id,fdc_id,nutrient_id,amount,data_points,derivation_id,min,max,median,footnote,min_year_acquired
0,2201847,319877,1051,56.30,1.0,1.0,NaN,NaN,NaN,NaN,NaN
1,2201845,319877,1002,1.28,1.0,1.0,NaN,NaN,NaN,NaN,NaN
2,2201846,319877,1004,19.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN
3,2201844,319877,1007,1.98,1.0,1.0,NaN,NaN,NaN,NaN,NaN
4,2201852,319878,1091,188.00,1.0,1.0,NaN,NaN,NaN,NaN,NaN


,id,name,unit_name,nutrient_nbr,rank
0,2047,Energy (Atwater General Factors),KCAL,957.0,280.0
1,2048,Energy (Atwater Specific Factors),KCAL,958.0,290.0
2,1001,Solids,G,201.0,200.0
3,1002,Nitrogen,G,202.0,500.0
4,1003,Protein,G,203.0,600.0


,id,code,description
0,1,100,Dairy and Egg Products
1,2,200,Spices and Herbs
2,3,300,Baby Foods
3,4,400,Fats and Oils
4,5,500,Poultry Products


,fdc_id,NDB_number,footnote
0,321358,16158,NaN
1,321360,100147,NaN
2,321611,11056,NaN
3,323121,7022,NaN
4,323294,12563,Other phytosterols = 34.67 mg/100g


In [ ]:
# Build an empty matrix template:
# first column -> unique fdc_id 
# following columns -> unique nutrient_id
# keep only fdc_id values that exist in foundation_food
foundation_fdc_ids = set(foundation_food["fdc_id"].dropna().unique())
unique_fdc_id = sorted(
    food_nutrient.loc[food_nutrient["fdc_id"].isin(foundation_fdc_ids), "fdc_id"]
    .dropna()
    .unique()
)
unique_nutrient_id = sorted(food_nutrient["nutrient_id"].dropna().unique())
nutrient_matrix = pd.DataFrame({"fdc_id": unique_fdc_id}).reindex( 
    columns=["fdc_id", *unique_nutrient_id] # old columns (fdc_id) listed so kept, new columns (*unique_nutrient_id) added and filled with NaN
)

# Add description and food category (food.csv) to the matrix   
food_meta = food[["fdc_id", "description", "food_category_id"]].drop_duplicates(
    subset=["fdc_id"]
)
nutrient_matrix = nutrient_matrix.merge(food_meta, on="fdc_id", how="left") # during merge, left is nutrient_matrix, right is food_meta

# Add food category description (food_category.csv) to the matrix
category_lookup = food_category.rename(columns={"id": "food_category_id"})[
    ["food_category_id", "description"]
].rename(columns={"description": "food_category_description"})
nutrient_matrix = nutrient_matrix.merge(category_lookup, on="food_category_id", how="left")

nutrient_matrix = nutrient_matrix[
    [
        "fdc_id",
        "description",
        "food_category_id",
        "food_category_description",
        *unique_nutrient_id,
    ]
]

print(nutrient_matrix.shape)
display(nutrient_matrix.head())

(365, 232)


,fdc_id,description,food_category_id,food_category_description,1002,1003,1004,1005,1007,1008,...,2057,2058,2059,2060,2061,2062,2063,2065,2066,2069
0,321358,"Hummus, commercial",16.0,Legumes and Legume Products,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,321360,"Tomatoes, grape, raw",11.0,Vegetables and Vegetable Products,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,321611,"Beans, snap, green, canned, regular pack, drai...",11.0,Vegetables and Vegetable Products,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,323121,"Frankfurter, beef, unheated",7.0,Sausages and Luncheon Meats,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,323294,"Nuts, almonds, dry roasted, with salt added",12.0,Nut and Seed Products,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# Fill nutrient columns from food_nutrient (amount per fdc_id and nutrient_id)
_meta_cols = ["fdc_id", "description", "food_category_id", "food_category_description"]

_fn_subset = food_nutrient.loc[
    food_nutrient["fdc_id"].isin(nutrient_matrix["fdc_id"])
    & food_nutrient["nutrient_id"].isin(unique_nutrient_id)
]
_amounts_wide = _fn_subset.pivot_table(
    index="fdc_id",
    columns="nutrient_id",
    values="amount",
    aggfunc="first",
)
_amounts_wide = _amounts_wide.reindex(
    index=nutrient_matrix["fdc_id"],
    columns=unique_nutrient_id,
)

nutrient_matrix = pd.concat(
    [
        nutrient_matrix[_meta_cols].reset_index(drop=True),
        _amounts_wide.reset_index(drop=True),
    ],
    axis=1,
)

print(nutrient_matrix.shape)
display(nutrient_matrix.head())


(365, 232)


,fdc_id,description,food_category_id,food_category_description,1002,1003,1004,1005,1007,1008,...,2057,2058,2059,2060,2061,2062,2063,2065,2066,2069
0,321358,"Hummus, commercial",16.0,Legumes and Legume Products,1.18,7.35,17.10,14.90,1.97,229.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,321360,"Tomatoes, grape, raw",11.0,Vegetables and Vegetable Products,0.13,0.83,0.63,5.51,0.56,27.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,321611,"Beans, snap, green, canned, regular pack, drai...",11.0,Vegetables and Vegetable Products,0.17,1.04,0.39,4.11,0.89,21.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,323121,"Frankfurter, beef, unheated",7.0,Sausages and Luncheon Meats,1.87,11.70,28.00,2.89,2.74,314.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,323294,"Nuts, almonds, dry roasted, with salt added",12.0,Nut and Seed Products,3.94,20.40,57.80,16.20,3.47,620.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Rename nutrient_id columns using nutrient table: "name (unit_name)"
from collections import Counter

_meta_cols = ["fdc_id", "description", "food_category_id", "food_category_description"]
_nutrient_id_cols = [c for c in nutrient_matrix.columns if c not in _meta_cols]

_lookup = nutrient.set_index("id")[["name", "unit_name"]]

_id_to_base = {}
for nid in _nutrient_id_cols:
    key = int(nid)
    if key not in _lookup.index:
        _id_to_base[nid] = f"unknown nutrient [{nid}]"
        continue
    row = _lookup.loc[key]
    _id_to_base[nid] = f"{row['name']} ({row['unit_name']})"

_base_counts = Counter(_id_to_base.values())
_rename = {
    nid: (base if _base_counts[base] == 1 else f"{base} [{nid}]")
    for nid, base in _id_to_base.items()
}
nutrient_matrix = nutrient_matrix.rename(columns=_rename)

print(nutrient_matrix.shape)
display(nutrient_matrix.head())


(365, 232)


,fdc_id,description,food_category_id,food_category_description,Nitrogen (G),Protein (G),Total lipid (fat) (G),"Carbohydrate, by difference (G)",Ash (G),Energy (KCAL),...,Ergothioneine (MG),Beta-glucan (G),Vitamin D4 (UG),Ergosta-7-enol (MG),"Ergosta-7,22-dienol (MG)","Ergosta-5,7-dienol (MG)",Verbascose (G),Low Molecular Weight Dietary Fiber (LMWDF) (G),unknown nutrient [2066],Glutathione (MG)
0,321358,"Hummus, commercial",16.0,Legumes and Legume Products,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,321360,"Tomatoes, grape, raw",11.0,Vegetables and Vegetable Products,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,321611,"Beans, snap, green, canned, regular pack, drai...",11.0,Vegetables and Vegetable Products,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,323121,"Frankfurter, beef, unheated",7.0,Sausages and Luncheon Meats,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,323294,"Nuts, almonds, dry roasted, with salt added",12.0,Nut and Seed Products,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Drop columns that contain any NaN
nutrient_matrix = nutrient_matrix.dropna(axis=1, how="any")

print(nutrient_matrix.shape)
display(nutrient_matrix.head())

_nutrient_matrix_path = data_dir / "nutrient_matrix.csv"
nutrient_matrix.to_csv(_nutrient_matrix_path, index=False)
print(f"Saved: {_nutrient_matrix_path.resolve()}")


In [ ]:
_nutrient_matrix_path = data_dir / "nutrient_matrix.csv"
nutrient_matrix.to_csv(_nutrient_matrix_path, index=False)
print(f"Saved: {_nutrient_matrix_path.resolve()}")